In [7]:
!pip install -q sentence-transformers faiss-cpu wikipedia langchain langchain-community langchain-text-splitters

print("All installed!")

All installed!


In [8]:
import faiss
import numpy as np
import wikipedia
import pickle
import os
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All imports done!")

All imports done!


In [9]:
# This model converts text into vectors (embeddings)
# It understands meaning — not just keywords
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded!")
print("Embedding size:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded!
Embedding size: 384


/tmp/ipykernel_1366/1916878122.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding size:", model.get_sentence_embedding_dimension())


In [11]:
# Using hardcoded documents instead of Wikipedia API
# (Wikipedia API blocked on college networks)

documents = [
    {
        'topic': 'Albert Einstein',
        'text': 'Albert Einstein was a German-born theoretical physicist who developed the theory of relativity, one of the two pillars of modern physics. His work is also known for its influence on the philosophy of science. He is best known to the general public for his mass-energy equivalence formula E=mc2. Einstein received the Nobel Prize in Physics in 1921 for his discovery of the law of the photoelectric effect. Einstein was born on 14 March 1879 in Ulm, Germany. He died on 18 April 1955 in Princeton, New Jersey.',
        'url': 'https://en.wikipedia.org/wiki/Albert_Einstein'
    },
    {
        'topic': 'Alexander Graham Bell',
        'text': 'Alexander Graham Bell was a Scottish-American inventor, scientist, and engineer who is credited with inventing and patenting the first practical telephone in 1876. He founded the Bell Telephone Company in 1877. Bell was born on March 3, 1847, in Edinburgh, Scotland. His mother and wife were both deaf which influenced his work on hearing devices. He died on August 2, 1922.',
        'url': 'https://en.wikipedia.org/wiki/Alexander_Graham_Bell'
    },
    {
        'topic': 'Isaac Newton',
        'text': 'Sir Isaac Newton was an English mathematician, physicist, astronomer, and author who is widely recognised as one of the greatest mathematicians and physicists of all time. His book Philosophiae Naturalis Principia Mathematica, published in 1687, established classical mechanics. Newton formulated the laws of motion and universal gravitation. He was born on 4 January 1643 and died on 31 March 1727. Newton never married and had no children.',
        'url': 'https://en.wikipedia.org/wiki/Isaac_Newton'
    },
    {
        'topic': 'Marie Curie',
        'text': 'Marie Curie was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity. She was the first woman to win a Nobel Prize, the first person to win the Nobel Prize twice, and the only person to win the Nobel Prize in two different sciences — Physics in 1903 and Chemistry in 1911. She was born on 7 November 1867 in Warsaw, Poland and died on 4 July 1934.',
        'url': 'https://en.wikipedia.org/wiki/Marie_Curie'
    },
    {
        'topic': 'World War II',
        'text': 'World War II was a global war that lasted from 1939 to 1945. It involved the vast majority of the world\'s countries forming two opposing military alliances: the Allies and the Axis. Germany invaded Poland on 1 September 1939, triggering the war. The war ended in 1945 with the victory of the Allies. An estimated 70 to 85 million people perished, making it the deadliest conflict in human history.',
        'url': 'https://en.wikipedia.org/wiki/World_War_II'
    },
    {
        'topic': 'Mahatma Gandhi',
        'text': 'Mahatma Gandhi was an Indian lawyer, anti-colonial nationalist and political ethicist who employed nonviolent resistance to lead the successful campaign for India\'s independence from British rule. India gained independence on 15 August 1947. Gandhi was born on 2 October 1869 in Porbandar, India. He was assassinated on 30 January 1948 in New Delhi by Nathuram Godse.',
        'url': 'https://en.wikipedia.org/wiki/Mahatma_Gandhi'
    },
    {
        'topic': 'Titanic',
        'text': 'The RMS Titanic was a British ocean liner that sank on 15 April 1912 after striking an iceberg during her maiden voyage from Southampton, England to New York City. The ship sank in the North Atlantic Ocean. Of the estimated 2,224 passengers and crew aboard, more than 1,500 died. The Titanic was built by Harland and Wolff shipyard in Belfast. It was not sunk by a storm.',
        'url': 'https://en.wikipedia.org/wiki/Titanic'
    },
    {
        'topic': 'Moon landing',
        'text': 'The Apollo 11 mission was the first crewed mission to land on the Moon. It was launched by NASA on July 16, 1969. Neil Armstrong became the first person to step onto the lunar surface on July 20, 1969, followed by Buzz Aldrin. Michael Collins orbited the Moon in the command module. Armstrong\'s famous words were "That\'s one small step for man, one giant leap for mankind."',
        'url': 'https://en.wikipedia.org/wiki/Apollo_11'
    },
    {
        'topic': 'Artificial intelligence',
        'text': 'Artificial intelligence is intelligence demonstrated by machines, as opposed to natural intelligence displayed by animals including humans. AI research has been defined as the field of study of intelligent agents. Modern AI applications include natural language processing, computer vision, robotics, and machine learning. The term artificial intelligence was coined by John McCarthy in 1956.',
        'url': 'https://en.wikipedia.org/wiki/Artificial_intelligence'
    },
    {
        'topic': 'Python programming language',
        'text': 'Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation. Python is dynamically typed and garbage-collected. It supports multiple programming paradigms, including structured, object-oriented and functional programming. Python was created by Guido van Rossum and first released in 1991.',
        'url': 'https://en.wikipedia.org/wiki/Python_(programming_language)'
    },
    {
        'topic': 'Climate change',
        'text': 'Climate change refers to long-term shifts in global temperatures and weather patterns. Since the 1800s, human activities have been the main driver of climate change, primarily due to burning fossil fuels like coal, oil and gas. The effects include rising sea levels, more extreme weather events, melting ice caps, and disruption to ecosystems. The Paris Agreement of 2015 aims to limit global warming.',
        'url': 'https://en.wikipedia.org/wiki/Climate_change'
    },
    {
        'topic': 'DNA',
        'text': 'Deoxyribonucleic acid (DNA) is a polymer composed of two polynucleotide chains that coil around each other to form a double helix. DNA carries genetic instructions for the development, functioning, growth and reproduction of all known organisms. The structure of DNA was discovered by James Watson and Francis Crick in 1953. DNA is found in the nucleus of cells.',
        'url': 'https://en.wikipedia.org/wiki/DNA'
    },
    {
        'topic': 'Nelson Mandela',
        'text': 'Nelson Mandela was a South African anti-apartheid activist and politician who served as the first president of South Africa from 1994 to 1999. He was the country\'s first black head of state. Mandela was imprisoned for 27 years on Robben Island. He was awarded the Nobel Peace Prize in 1993. He was born on 18 July 1918 and died on 5 December 2013.',
        'url': 'https://en.wikipedia.org/wiki/Nelson_Mandela'
    },
    {
        'topic': 'Human brain',
        'text': 'The human brain is the central organ of the human nervous system. It consists of approximately 86 billion neurons. The brain controls most of the activities of the body. It has three main parts: the cerebrum, cerebellum, and brainstem. The cerebrum is the largest part and is responsible for thinking, memory, and language. The average adult human brain weighs about 1.3 kg.',
        'url': 'https://en.wikipedia.org/wiki/Human_brain'
    },
    {
        'topic': 'Solar system',
        'text': 'The Solar System consists of the Sun and the objects that orbit it. There are eight planets in the Solar System: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune. Pluto was reclassified as a dwarf planet in 2006. The Solar System formed about 4.6 billion years ago. Earth is the third planet from the Sun and the only known planet to harbour life.',
        'url': 'https://en.wikipedia.org/wiki/Solar_System'
    }
]

print(f"Total documents loaded: {len(documents)}")
for doc in documents:
    print(f"✅ Loaded: {doc['topic']}")

Total documents loaded: 15
✅ Loaded: Albert Einstein
✅ Loaded: Alexander Graham Bell
✅ Loaded: Isaac Newton
✅ Loaded: Marie Curie
✅ Loaded: World War II
✅ Loaded: Mahatma Gandhi
✅ Loaded: Titanic
✅ Loaded: Moon landing
✅ Loaded: Artificial intelligence
✅ Loaded: Python programming language
✅ Loaded: Climate change
✅ Loaded: DNA
✅ Loaded: Nelson Mandela
✅ Loaded: Human brain
✅ Loaded: Solar system


In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)

chunks = []
for doc in documents:
    splits = splitter.split_text(doc['text'])
    for split in splits:
        chunks.append({
            'text': split,
            'topic': doc['topic'],
            'url': doc['url']
        })

print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk:")
print(chunks[0]['text'])

Total chunks created: 40

Sample chunk:
Albert Einstein was a German-born theoretical physicist who developed the theory of relativity, one of the two pillars of modern physics. His work is also known for its influence on the philosophy of


In [13]:
print("Creating embeddings... (takes 1-2 minutes)")

texts = [chunk['text'] for chunk in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

print(f"\nEmbeddings shape: {embeddings.shape}")

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings, dtype=np.float32))

print(f"FAISS index built!")
print(f"Total vectors in index: {index.ntotal}")

Creating embeddings... (takes 1-2 minutes)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Embeddings shape: (40, 384)
FAISS index built!
Total vectors in index: 40


In [14]:
faiss.write_index(index, 'knowledge_base.index')

with open('chunks.pkl', 'wb') as f:
    pickle.dump(chunks, f)

print("Saved knowledge_base.index ✅")
print("Saved chunks.pkl ✅")

Saved knowledge_base.index ✅
Saved chunks.pkl ✅


In [15]:
def retrieve_evidence(question, top_k=3):
    # Step 1: Embed the question
    question_embedding = model.encode([question])

    # Step 2: Search FAISS for most similar chunks
    distances, indices = index.search(
        np.array(question_embedding, dtype=np.float32),
        top_k
    )

    # Step 3: Return top-k results
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'text': chunks[idx]['text'],
            'topic': chunks[idx]['topic'],
            'url': chunks[idx]['url'],
            'score': float(distances[0][i])
        })

    return results

print("Retrieval function ready! ✅")

Retrieval function ready! ✅


In [16]:
test_questions = [
    "Who invented the telephone?",
    "What did Einstein discover?",
    "When did World War 2 end?",
    "Who was the first person on the moon?",
    "What did Newton discover about gravity?"
]

for question in test_questions:
    print(f"\n{'='*50}")
    print(f"Question: {question}")
    print(f"Top evidence retrieved:")

    results = retrieve_evidence(question, top_k=2)
    for i, result in enumerate(results):
        print(f"\n  [{i+1}] Topic: {result['topic']}")
        print(f"       Text: {result['text'][:150]}...")
        print(f"       Score: {result['score']:.4f}")


Question: Who invented the telephone?
Top evidence retrieved:

  [1] Topic: Alexander Graham Bell
       Text: Alexander Graham Bell was a Scottish-American inventor, scientist, and engineer who is credited with inventing and patenting the first practical telep...
       Score: 0.6163

  [2] Topic: Alexander Graham Bell
       Text: He founded the Bell Telephone Company in 1877. Bell was born on March 3, 1847, in Edinburgh, Scotland. His mother and wife were both deaf which influe...
       Score: 0.8715

Question: What did Einstein discover?
Top evidence retrieved:

  [1] Topic: Albert Einstein
       Text: Albert Einstein was a German-born theoretical physicist who developed the theory of relativity, one of the two pillars of modern physics. His work is ...
       Score: 0.7182

  [2] Topic: Albert Einstein
       Text: on the philosophy of science. He is best known to the general public for his mass-energy equivalence formula E=mc2. Einstein received the Nobel Prize ...
       Scor

In [17]:
from google.colab import files
files.download('knowledge_base.index')
files.download('chunks.pkl')
print("Downloaded! ✅")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded! ✅
